In [1]:
import pandas as pd

import numpy as np
import json

In [2]:
import json
with open("../citation.json", 'r') as f:
    citation = json.load(f)
citation

{'doi': 'https://doi.org/10.1038/s41467-021-25968-8',
 'citation': 'Gautam, P., Hamashima, K., Chen, Y. et al. Multi-species single-cell transcriptomic analysis of ocular compartment regulons. Nat Commun 12, 5675 (2021).',
 'tables': ['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8478974/bin/41467_2021_25968_MOESM5_ESM.xlsx']}

## Define the metadata

In [3]:
organism = "homo_sapiens"
cell_source = "eye"
cell_state = None

reference = "GRCh38"

table_name = "clean_degs.xlsx" # local name

## Read in file

In [4]:
excel_1 = pd.read_excel(table_name, skiprows = 1, sheet_name = "ChoroidSclera")
excel_2 = pd.read_excel(table_name, skiprows = 1, sheet_name = "Cornea")
excel_3 = pd.read_excel(table_name, skiprows = 1, sheet_name = "IrisCiliaryBody")


df_1 = excel_1.rename(columns={"cluster": "celltype"})
df_2 = excel_2.rename(columns={"cluster": "celltype"})
df_3 = excel_3.rename(columns={"cluster": "celltype"})

# Decided to unify the cell types from the 3 tissues within the eye because otherwise many genes were repeated.
# df_1["celltype"] = "ChoroidSclera_" + df_1["celltype"]
# df_2["celltype"] = "Cornea_" + df_2["celltype"]
# df_3["celltype"] = "IrisCiliaryBody" + df_3["celltype"]

df = pd.concat([df_1, df_2, df_3])


In [5]:
df.head()

,Unnamed: 0,p_val,avg_logFC,pct.1,pct.2,p_val_adj,celltype,gene
0,IGFBP5,1.917143e-260,2.740738,0.981,0.809,3.479997e-256,Fibroblasts,IGFBP5
1,MGP,8.043193e-214,1.760688,1.000,0.992,1.460000e-209,Fibroblasts,MGP
2,RARRES1,2.216642e-198,2.315193,0.905,0.622,4.023649e-194,Fibroblasts,RARRES1
3,FBLN1,4.029294e-198,1.967178,0.894,0.596,7.313975e-194,Fibroblasts,FBLN1
4,C1R,3.932635e-193,1.828830,0.872,0.491,7.138518e-189,Fibroblasts,C1R


## Unfiltered

In [6]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"celltype": "cell_type_label", "gene": "feature_name"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None

unfiltered_df.drop(['p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj', "Unnamed: 0"], axis=1, inplace=True) 
result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

## Save unfiltered

In [7]:
with open("evidence_unfiltered.json", 'w') as f:
    json.dump(result, f, indent=4)

## Perform the filtering

In [6]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "celltype"
col_rank = "pct.1"

In [7]:
min_mean = 100
max_pval = 1e-10
min_lfc = 0.7
max_gene_shares = 4
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")


# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

if col_rank:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)
else:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)]

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [8]:
markers[col_ct].value_counts()

celltype
Fibroblasts                          20
Putative stem cells                  20
Smooth muscle cells                  20
Ribosomal genes-hi fibroblasts       20
Melanocytes                          20
Conjunctival cells                   20
Activated T cells                    20
ELF3-hi corneal epithelial cells     20
TGFBI-hi corneal epithelial cells    20
Ciliary body endothelial cells       20
Monocytes                            20
Cytotoxic T cells                    20
WIF1-hi fibroblasts                  20
Choroid endothelial cells            20
Schwann cells                        20
Ciliary body cells                   20
MGP-hi fibroblasts                   20
MEG3-hi fibroblasts                  20
Pigmented ciliary body cells         18
Corneal fibroblasts                  18
COL9A1-hi ciliary body cells         14
CRYAA-hi ciliary body cells          10
Name: count, dtype: int64

In [9]:
if col_rank:
    print(markers.groupby(col_ct)[col_rank].mean().sort_values())

celltype
Corneal fibroblasts                  0.778889
COL9A1-hi ciliary body cells         0.780000
MEG3-hi fibroblasts                  0.840950
Schwann cells                        0.842050
MGP-hi fibroblasts                   0.852000
Ciliary body cells                   0.885700
Pigmented ciliary body cells         0.904611
Fibroblasts                          0.919400
Choroid endothelial cells            0.928700
CRYAA-hi ciliary body cells          0.951500
WIF1-hi fibroblasts                  0.957800
Cytotoxic T cells                    0.960750
Putative stem cells                  0.966700
Ciliary body endothelial cells       0.970850
TGFBI-hi corneal epithelial cells    0.992450
Activated T cells                    0.993400
ELF3-hi corneal epithelial cells     0.993600
Conjunctival cells                   0.994700
Melanocytes                          0.998850
Ribosomal genes-hi fibroblasts       0.999400
Monocytes                            1.000000
Smooth muscle cells      

## Find Ensembl ID


In [10]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [11]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [12]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json

In [13]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
df["feature_identifier"] = df["feature_identifier"].apply(run)
result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

In [14]:
save

[{'cell_type_label': 'Corneal fibroblasts',
  'gene': 'IGFBP5',
  'organism': 'homo_sapiens',
  'cell_source': 'eye',
  'cell_state': None,
  'gene_id': 'ENSG00000115461'},
 {'cell_type_label': 'Corneal fibroblasts',
  'gene': 'SAA1',
  'organism': 'homo_sapiens',
  'cell_source': 'eye',
  'cell_state': None,
  'gene_id': 'ENSG00000173432'},
 {'cell_type_label': 'Corneal fibroblasts',
  'gene': 'CCL2',
  'organism': 'homo_sapiens',
  'cell_source': 'eye',
  'cell_state': None,
  'gene_id': 'ENSG00000108691'},
 {'cell_type_label': 'Corneal fibroblasts',
  'gene': 'NNMT',
  'organism': 'homo_sapiens',
  'cell_source': 'eye',
  'cell_state': None,
  'gene_id': 'ENSG00000166741'},
 {'cell_type_label': 'Corneal fibroblasts',
  'gene': 'SERPINE2',
  'organism': 'homo_sapiens',
  'cell_source': 'eye',
  'cell_state': None,
  'gene_id': 'ENSG00000135919'},
 {'cell_type_label': 'Corneal fibroblasts',
  'gene': 'IGFBP4',
  'organism': 'homo_sapiens',
  'cell_source': 'eye',
  'cell_state': None,

## Save evidence

In [15]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 